# What you can sweep

`run_experiment` takes a dictionary of parameters and turns it into a table. Every key in that
dictionary is either **held** (`Fixed`) or **varied** (`Sweep`), and this notebook is the catalogue:
what you are allowed to put there, what each one means, and the handful that behave in surprising
ways.

Most of it runs no simulation at all — a declaration can be expanded into its conditions without
flying anything, which is the cheapest way to check you have written what you meant.

In [1]:
import dataclasses

import pandas as pd

from opencdarr import (
    MC,
    MVP,
    VO,
    SMALL_FIXEDWING,
    Airframe,
    Fixed,
    FixedWing,
    GnssNavigation,
    M600,
    Methods,
    PastCPA,
    StateBased,
    Sweep,
    WindField,
    run_experiment,
)
from opencdarr import experiment as E
from opencdarr.config import load_config
from opencdarr.experiment import expand

base = load_config("../../configs/pairwise.yaml")
print(f"{len(E._KNOWN_KEYS)} parameters can be declared")

29 parameters can be declared


## The two roles

There are exactly two, and no third.

- **`Fixed(v)`** — held at `v` for every condition. It overrides whatever the base config says.
- **`Sweep([a, b, c])`** — an output axis. One condition per level, and the levels become a column
  in the results table.

The conditions are the cross-product of every `Sweep`, in declaration order. Nothing is run to work
that out, so you can check a declaration before committing to the compute:

In [2]:
conditions = expand({
    "dpsi": Sweep([10.0, 45.0, 90.0]),
    "pos_ci95": Sweep([10.0, 30.0]),
    "dcpa": Fixed(0.0),          # held: appears in every condition, in no column
})
print(f"{len(conditions)} conditions from 3 x 2 levels\n")
for c in conditions:
    print(" ", c.label)

6 conditions from 3 x 2 levels

  {'dpsi': 10.0, 'pos_ci95': 10.0}
  {'dpsi': 10.0, 'pos_ci95': 30.0}
  {'dpsi': 45.0, 'pos_ci95': 10.0}
  {'dpsi': 45.0, 'pos_ci95': 30.0}
  {'dpsi': 90.0, 'pos_ci95': 10.0}
  {'dpsi': 90.0, 'pos_ci95': 30.0}


Note what `Fixed` does *not* do: it never becomes a column, because a column that is the same in
every row carries no information. It is still applied to every run.

### `build`: numbers in the table, objects in the run

Sweeping a *component* is awkward if the levels are objects — the table fills with `repr`s. `build`
maps each level onto the value the run actually needs, so you can sweep something readable:

In [3]:
by_name = expand({"resolver": Sweep(["MVP", "VO"],
                                    build={"MVP": MVP(1.05), "VO": VO(1.05)}.__getitem__)})
print("labels the table will show:", [c.label for c in by_name])
print("what the run receives:     ", [type(c.get("resolver")).__name__ for c in by_name])

# the same trick over a parameter of one component
by_margin = expand({"resolver": Sweep([1.05, 1.4], build=lambda m: MVP(margin=m), name="margin")})
print("\nsweeping a resolver's own parameter:", [c.label for c in by_margin])

labels the table will show: [{'resolver': 'MVP'}, {'resolver': 'VO'}]
what the run receives:      ['MVP', 'VO']

sweeping a resolver's own parameter: [{'margin': 1.05}, {'margin': 1.4}]


## The catalogue

Everything you may declare, grouped by what it controls.

In [4]:
CATALOGUE = {
    # the encounter as it is sampled
    "speed": ("scenario", "ownship ground speed", "m/s"),
    "gs_intr": ("geometry", "intruder ground speed (defaults to the ownship's)", "m/s"),
    "dpsi": ("geometry", "crossing angle between the two tracks", "deg"),
    "dcpa": ("geometry", "miss distance the encounter is built to reach", "m"),
    "side": ("geometry", "which side the intruder passes", "+1 / -1"),
    "dcpa_max": ("scenario", "upper bound when dcpa is drawn rather than pinned", "m"),
    "tlos": ("scenario", "time from spawn to protected-zone entry", "s"),
    # what the aircraft know about themselves
    "pos_ci95": ("scenario", "position accuracy, 95% radial", "m"),
    "vel_ci95": ("scenario", "velocity accuracy, 95% radial", "m/s"),
    "pos_ci95_declared": ("scenario", "position accuracy the broadcast claims", "m"),
    "vel_ci95_declared": ("scenario", "velocity accuracy the broadcast claims", "m/s"),
    # the separation problem
    "rpz": ("conflict", "protected zone radius", "m"),
    "t_lookahead": ("conflict", "how far ahead the detector predicts", "s"),
    # the numerics and the transmit clock
    "dt": ("simulation", "integration timestep", "s"),
    "t_max": ("simulation", "hard stop on one encounter", "s"),
    "done_timeout": ("simulation", "how long clear before a run ends", "s"),
    "broadcast_interval": ("simulation", "transmit period", "s"),
    "broadcast_jitter": ("simulation", "per-transmission dither", "s"),
    "broadcast_random_phase": ("simulation", "unsynchronised transmitters", "bool"),
    # the swappable pieces
    "detector": ("component", "ConflictDetector", "object"),
    "resolver": ("component", "ConflictResolver", "object"),
    "recovery": ("component", "RecoveryCriterion", "object"),
    "navigation": ("component", "NavigationModel", "object"),
    "communication": ("component", "CommunicationModel", "object"),
    "surveillance": ("component", "SurveillanceModel", "object"),
    "kinematics": ("component", "Kinematics, shared by both aircraft", "object"),
    "perf": ("component", "Performance, shared by both aircraft", "object"),
    "airframes": ("component", "one Airframe per aircraft (a mixed fleet)", "list"),
    "wind": ("component", "steady uniform wind", "WindField"),
}

# this table is checked against the library rather than trusted: a parameter added to
# run_experiment and not documented here fails the notebook instead of going unnoticed.
missing = E._KNOWN_KEYS - set(CATALOGUE)
extra = set(CATALOGUE) - E._KNOWN_KEYS
assert not missing, f"declarable but undocumented: {sorted(missing)}"
assert not extra, f"documented but not declarable: {sorted(extra)}"

frame = pd.DataFrame(
    [{"parameter": k, "group": g, "meaning": m, "unit": u} for k, (g, m, u) in CATALOGUE.items()]
)
frame.sort_values(["group", "parameter"]).set_index(["group", "parameter"])

meaning  \
group      parameter                                                                   
component  airframes                       one Airframe per aircraft (a mixed fleet)   
           communication                                          CommunicationModel   
           detector                                                 ConflictDetector   
           kinematics                            Kinematics, shared by both aircraft   
           navigation                                                NavigationModel   
           perf                                 Performance, shared by both aircraft   
           recovery                                                RecoveryCriterion   
           resolver                                                 ConflictResolver   
           surveillance                                            SurveillanceModel   
           wind                                                  steady uniform wind   
conflict   rpz                                                 protected zone radius   
           t_lookahead                           how far ahead the detector predicts   
geometry   dcpa                        miss distance the encounter is built to reach   
           dpsi                                crossing angle between the two tracks   
           gs_intr                 intruder ground speed (defaults to the ownship's)   
           side                                       which side the intruder passes   
scenario   dcpa_max                upper bound when dcpa is drawn rather than pinned   
           pos_ci95                                    position accuracy, 95% radial   
           pos_ci95_declared                  position accuracy the broadcast claims   
           speed                                                ownship ground speed   
           tlos                              time from spawn to protected-zone entry   
           vel_ci95                                    velocity accuracy, 95% radial   
           vel_ci95_declared                  velocity accuracy the broadcast claims   
simulation broadcast_interval                                        transmit period   
           broadcast_jitter                                  per-transmission dither   
           broadcast_random_phase                        unsynchronised transmitters   
           done_timeout                             how long clear before a run ends   
           dt                                                   integration timestep   
           t_max                                          hard stop on one encounter   

                                        unit  
group      parameter                          
component  airframes                    list  
           communication              object  
           detector                   object  
           kinematics                 object  
           navigation                 object  
           perf                       object  
           recovery                   object  
           resolver                   object  
           surveillance               object  
           wind                    WindField  
conflict   rpz                             m  
           t_lookahead                     s  
geometry   dcpa                            m  
           dpsi                          deg  
           gs_intr                       m/s  
           side                      +1 / -1  
scenario   dcpa_max                        m  
           pos_ci95                        m  
           pos_ci95_declared               m  
           speed                         m/s  
           tlos                            s  
           vel_ci95                      m/s  
           vel_ci95_declared             m/s  
simulation broadcast_interval              s  
           broadcast_jitter                s  
           broadcast_random_phase       bool  
           done_timeout         

Anything not on that list is refused at declaration time, with the list in the message — so a typo
or a parameter you hoped existed fails immediately rather than being silently ignored:

In [5]:
for wrong in ("crossing_angle", "margin", "n_encounters"):
    try:
        expand({wrong: Sweep([1.0])})
    except ValueError as e:
        print(f"{wrong:<16} -> {str(e)[:66]}...")

crossing_angle   -> unknown parameter(s) ['crossing_angle']. Declarable: ['airframes',...
margin           -> unknown parameter(s) ['margin']. Declarable: ['airframes', 'broadc...
n_encounters     -> unknown parameter(s) ['n_encounters']. Declarable: ['airframes', '...


`n_encounters` is the interesting refusal: it is a real setting, but it belongs to the **backend**
(`MC(n_encounters=...)`), not to the declaration. Sample size is not a property of the scenario.

## A sweep that actually runs

Nothing above flew anything. Here is a small real one — a scalar axis and a component axis crossed,
four conditions:

In [6]:
cfg = dataclasses.replace(base, scenario=dataclasses.replace(
    base.scenario, speed=10.0, tlos=180.0, pos_ci95=20.0, vel_ci95=2.0))

RESOLVERS = {"MVP": MVP(1.05), "VO": VO(1.05)}
res = run_experiment(
    {"dpsi": Sweep([10.0, 45.0]),
     "resolver": Sweep(list(RESOLVERS), build=RESOLVERS.__getitem__, name="resolver"),
     "dcpa": Fixed(0.0), "pos_ci95": Fixed(20.0), "vel_ci95": Fixed(2.0)},
    methods=Methods(detector=StateBased(), recovery=PastCPA(), navigation=GnssNavigation(),
                    perf=M600),
    backend=MC(n_encounters=200), base_config=cfg, seed=0, n_jobs=4,
)
res.frame()[["dpsi", "resolver", "p_los", "p_los_lo", "p_los_hi", "median_min_sep"]].round(4)

,dpsi,resolver,p_los,p_los_lo,p_los_hi,median_min_sep
0,10.0,MVP,0.030,0.0138,0.0639,69.5573
1,10.0,VO,0.200,0.1505,0.2609,70.7334
2,45.0,MVP,0.000,0.0000,0.0188,229.5252
3,45.0,VO,0.005,0.0009,0.0278,230.5352


## Eight axes at once

Nothing stops you sweeping several parameters together — the conditions are just their
cross-product. Here are eight, two levels each:

| axis | levels | what it varies |
|---|---|---|
| `speed` | 13, 17 m/s | how fast the ownship flies |
| `gs_intr` | 13, 17 m/s | how fast the intruder flies |
| `dpsi` | 45°, 90° | the crossing angle |
| `pos_ci95` | 10, 30 m | how well each aircraft knows where it is |
| `vel_ci95` | 1, 3 m/s | how well it knows how fast it is going |
| `dcpa` | 0, 25 m | how close the encounter is built to come |
| `resolver` | MVP, VO | which avoidance algorithm |
| `airframes` | copter v copter, copter v plane | whether the fleet is mixed |

The last one is why the speeds start at 13 rather than 10. A multirotor will fly at any speed in
`[-18, 18]` m/s; the small fixed-wing **stalls below 12**. There is no single pair of speed levels
that suits both fleets *and* includes 10 m/s, so the levels have to sit inside both envelopes.

Try it with 10 m/s in there and the run is refused rather than flown:

In [7]:
RESOLVERS = {"MVP": MVP(1.05), "VO": VO(1.05)}
FLEETS = {
    "copter v copter": [Airframe(M600), Airframe(M600)],
    "copter v plane": [Airframe(M600), Airframe(SMALL_FIXEDWING, FixedWing())],
}

declared = {
    "speed": Sweep([13.0, 15.0]),        # ownship
    "gs_intr": Sweep([13.0, 15.0]),      # intruder
    "dpsi": Sweep([45.0, 90.0]),
    "pos_ci95": Sweep([10.0, 30.0]),
    "vel_ci95": Sweep([1.0, 3.0]),
    "dcpa": Sweep([0.0, 25.0]),
    "resolver": Sweep(list(RESOLVERS), build=RESOLVERS.__getitem__, name="resolver"),
    "airframes": Sweep(list(FLEETS), build=FLEETS.__getitem__, name="fleet"),
}

kw = dict(
    # navigation is required because pos_ci95 / vel_ci95 are swept -- something has to draw the
    # error from them. perf=M600 is the default airframe; the `airframes` axis replaces it per
    # condition, so the two do not clash.
    methods=Methods(detector=StateBased(), recovery=PastCPA(),
                    navigation=GnssNavigation(), perf=M600),
    backend=MC(n_encounters=5),
    base_config=dataclasses.replace(base, scenario=dataclasses.replace(base.scenario, tlos=180.0)),
    seed=0, n_jobs=4,
)

try:
    run_experiment(declared, **kw)
except ValueError as e:
    print(f"refused: {e}")

In [8]:
res = run_experiment(declared, **kw)

In [9]:
RESOLVERS = {"MVP": MVP(1.05), "VO": VO(1.05)}
FLEETS = {
    "copter v copter": [Airframe(M600), Airframe(M600)],
    "copter v plane": [Airframe(M600), Airframe(SMALL_FIXEDWING, FixedWing())],
}

declared = {
    "speed": Sweep([10.0, 15.0]),        # ownship
    "gs_intr": Sweep([10.0, 15.0]),      # intruder
    "dpsi": Sweep([45.0, 90.0]),
    "pos_ci95": Sweep([10.0, 30.0]),
    "vel_ci95": Sweep([1.0, 3.0]),
    "dcpa": Sweep([0.0, 25.0]),
    "resolver": Sweep(list(RESOLVERS), build=RESOLVERS.__getitem__, name="resolver"),
    "airframes": Sweep(list(FLEETS), build=FLEETS.__getitem__, name="fleet"),
}

kw = dict(
    # navigation is required because pos_ci95 / vel_ci95 are swept -- something has to draw the
    # error from them. perf=M600 is the default airframe; the `airframes` axis replaces it per
    # condition, so the two do not clash.
    methods=Methods(detector=StateBased(), recovery=PastCPA(),
                    navigation=GnssNavigation(), perf=M600),
    backend=MC(n_encounters=5),
    base_config=dataclasses.replace(base, scenario=dataclasses.replace(base.scenario, tlos=180.0)),
    seed=0, n_jobs=4,
)

try:
    run_experiment(declared, **kw)
except ValueError as e:
    print(f"refused: {e}")

refused: initial ground speed 10.0 m/s for 'INT' is outside its envelope [12.0, 25.0] m/s. In a mixed fleet, set each aircraft's speed for its own airframe (the sampler's `speed` / `gs_intr`).


That is the mixed-fleet guard: an aircraft cannot spawn at a speed its own airframe cannot fly. Note
**the whole call aborts**, not just the offending condition — the check happens inside the run, so
there is no per-condition skip, and with a large cross-product you find out about one bad level at a
time. Checking your levels against every envelope you are sweeping over is cheaper than discovering
it 200 conditions in.

Move both speed axes inside both envelopes and it completes:

In [ ]:
declared["speed"] = Sweep([13.0, 17.0])
declared["gs_intr"] = Sweep([13.0, 17.0])

everything = run_experiment(declared, **kw)

print(f"{len(everything)} conditions = 2^8")
print(f"axes in declaration order: {list(everything.axes)}")
everything.frame()[["fleet", "speed", "gs_intr", "dpsi", "pos_ci95", "vel_ci95", "dcpa",
                    "resolver", "p_los", "p_los_hi", "n_los", "median_min_sep"]].round(3).head(12)

Every swept axis becomes a column, in declaration order, and the metrics follow. The full frame is
256 rows; the head shows the first twelve.

**Watch how fast this grows.** Eight binary axes is 256 conditions, which took about 18 seconds at
five encounters each. At a realistic 500 encounters per condition that is 128 000 encounters rather
than 1280 — the same declaration, two and a half orders of magnitude more compute.

Two habits worth having:

- **Check the declaration with `expand()` first.** It costs nothing and tells you the condition
  count before you commit to the compute.
- **Turn `cache=True` on** for anything you might extend. Each condition is keyed separately, so
  adding a level later re-runs only the new cells.

And read the `p_los_hi` column before drawing any conclusion. At five encounters, a cell showing
`p_los = 0.0` has an upper bound near 0.43 — it says almost nothing, and both fleets average
*exactly* zero. The declaration is what this section demonstrates; the sample size is a placeholder.

## Four that do not behave the way they look

**`pos_ci95` and `vel_ci95` do nothing without a navigation model.** They are numbers stamped on
every aircraft; a `NavigationModel` is what *draws an error from them*. Declare a noise sweep with no
navigation and every cell would be identical — a publishable-looking null result produced by a
no-op. The runner refuses that declaration instead:

In [ ]:
try:
    run_experiment({"pos_ci95": Sweep([0.0, 20.0]), "vel_ci95": Fixed(2.0)},
                   methods=Methods(detector=StateBased(), resolver=MVP(1.05), perf=M600),
                   backend=MC(n_encounters=10), base_config=cfg, seed=0)
except ValueError as e:
    print(str(e)[:300])

**`broadcast_random_phase` needs `dt` well below `broadcast_interval`.** The phase offsets each
aircraft's transmit clock, but a clock is only read on the timestep grid — so the number of
distinguishable phases is `broadcast_interval / dt`. At the default 1 s interval with `dt = 1.0`,
that is **one**: every aircraft still fires on the same ticks, which is exactly the synchronised case
the option exists to avoid. There is no guard on this one; it is on you.

In [ ]:
import numpy as np
from opencdarr.cns.broadcast import BroadcastSchedule

g = np.random.default_rng(0)
phases = list(g.uniform(0.0, 1.0, 200))            # 200 aircraft, 200 random phases
print(f"{'dt [s]':>7} {'distinct firing patterns':>26}")
for dt in (1.0, 0.5, 0.1, 0.05):
    sched = BroadcastSchedule(interval=1.0, phase=phases)
    clock, t, seen = sched.initial(200), 0.0, {i: [] for i in range(200)}
    while t <= 5.0 + 1e-9:
        for i in sched.due(clock, t):
            seen[i].append(round(t, 4))
            clock[i] = sched.advance(clock[i], None)
        t += dt
    print(f"{dt:>7} {len({tuple(v) for v in seen.values()}):>26}")

**`kinematics` and `perf` are shared; `airframes` is per aircraft.** The first two set one airframe
for everyone, which is right when both aircraft are the same. A mixed fleet uses `airframes`
instead — and the two spellings are mutually exclusive, so saying both is refused rather than
resolved in some undocumented order. See the [mixed fleet notebook](mixed_fleet.ipynb).

**`dcpa_max` only matters when `dcpa` is drawn.** Pin `dcpa` and the bound is irrelevant; leave it
out and the miss distance is drawn from `U(0, dcpa_max)`. The same pattern holds for `dpsi` and
`side`: pinning one turns off its draw, and the config field that shaped that draw stops mattering.

## Two things that are not swept here

**The sample size** belongs to the backend: `MC(n_encounters=...)` or
`IPS(shells=..., n_particles=..., reps=...)`. Sweeping it would be measuring the estimator rather
than the system.

**A distribution over a parameter.** A `Sweep` gives you `P(LoS | dpsi)` — a response curve, one
number per level. If you want `P(LoS)` averaged over a distribution of crossing angles, sweep the
axis and weight the *counts* yourself. Weighting the per-condition rates instead is the mistake that
`combine_ipr` exists to warn about, because a cell with few encounters would count as heavily as one
with many.